In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import numpy as np
import tf_keras as tf_keras
import matplotlib.pyplot as plt
import tensorflow as tf
from PIL import Image
import io

IMG_SIZE = 224 # Image size for processing
classes = ['Bean', 'Bitter_Gourd', 'Bottle_Gourd', 'Brinjal', 'Broccoli', 'Cabbage',
          'Capsicum', 'Carrot', 'Cauliflower', 'Cucumber', 'Papaya', 'Potato',
          'Pumpkin', 'Radish', 'Tomato']
model_path = 'model_2.keras'

# Load the model
def load_model():
    print("Loading model...") 
    try:
        model = tf_keras.models.load_model(model_path,custom_objects={'data_augmentation': None})
        print("Model loaded successfully!")
        return model
    except Exception as e:
        print(f"Error loading model: {e}")
        return None

model = load_model()

# Image preprocessing function
def load_and_prep_image(image, img_shape=224, scale=True):
    """Loads an image, turns it into a tensor and reshapes it."""
    img = tf.image.resize(image, [img_shape, img_shape])
    return img / 255. if scale else img # Normalize pixel values

# Output Widget for Logs and Display
output = widgets.Output()

# Function to process uploaded image and make predictions
def predict_uploaded_image(change):
    with output:
        clear_output(wait=True)
        if model is None:
            print("Model failed to load. Please check the file path.")
            return
            
        uploaded_file = change['new']
        if not uploaded_file:
            print("No file uploaded. Please upload an image.")
            return
            
        try:
            # Read and process image
            file_content = uploaded_file[0].content
            image = Image.open(io.BytesIO(file_content)).convert("RGB")
            img_array = np.array(image)
          
            img_tensor = tf.convert_to_tensor(img_array, dtype=tf.float32)
            img_tensor = load_and_prep_image(img_tensor, scale=False)
            img_array = tf.expand_dims(img_tensor, axis=0) # Add batch dimension

            # Predict
            pred_prob = model.predict(img_array)
            pred_class = classes[np.argmax(pred_prob)]

            # Display image
            plt.figure(figsize=(7, 7))
            plt.imshow(image)
            plt.title(f"Predicted: {pred_class}, Prob: {np.max(pred_prob):.2f}", color="blue")
            plt.axis(False)
            plt.show()

        except Exception as e:
            print(f"Error during prediction: {e}")

upload = widgets.FileUpload(accept='image/*', multiple=False)
upload.observe(predict_uploaded_image, names='value')

display(upload, output)